In [ ]:

# **Cell 1**
from google.colab import drive
import os, re, json, math, random, subprocess, shutil, glob, sys
from pathlib import Path
from PIL import Image

drive.mount('/content/drive', force_remount=False)
FOLDER='/content/drive/MyDrive/AIgenerated'
AUDIO_PATH=os.path.join(FOLDER,'trimmed_audio.m4a')
SONG_MAP_PATH=os.path.join(FOLDER,'song_map.json')
ASSETS_ROOT=os.path.join(FOLDER,'assets')
FONTS_ROOT=os.path.join(ASSETS_ROOT,'fonts')
EMOJI_ROOT=os.path.join(ASSETS_ROOT,'emoji')
OUTPUT_ROOT=os.path.join(FOLDER,'stage2_outputs')
WORK_ROOT=os.path.join(FOLDER,'_stage2_work')
os.makedirs(OUTPUT_ROOT,exist_ok=True); os.makedirs(WORK_ROOT,exist_ok=True)
print('✅ Drive mounted; Stage 2 workspace ready.')

Mounted at /content/drive
✅ Drive mounted; Stage 2 workspace ready.


In [ ]:

# **Cell 2**
if not os.path.isfile(AUDIO_PATH): raise FileNotFoundError(AUDIO_PATH)
if not os.path.isfile(SONG_MAP_PATH): raise FileNotFoundError(SONG_MAP_PATH)
with open(SONG_MAP_PATH,'r',encoding='utf-8') as f: SONG_MAP=json.load(f)
LINES=SONG_MAP.get('lines',[])
if not LINES: raise RuntimeError('No lines in song_map.json')
probe=subprocess.run(['ffprobe','-v','error','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',AUDIO_PATH],capture_output=True,text=True,check=True)
AUDIO_DURATION=float(probe.stdout.strip())
print('STAGE 1 VALIDATION'); print('==================')
print('Lines:',len(LINES)); print(f'Audio duration: {AUDIO_DURATION:.3f}s')
print('Source timing map loaded; no alignment/Whisper work will run.')

STAGE 1 VALIDATION
Lines: 4
Audio duration: 9.060s
Source timing map loaded; no alignment/Whisper work will run.


In [ ]:
# **Cell 3**
def natural_key(p):
    return [int(x) if x.isdigit() else x.lower() for x in re.split(r'(\d+)',Path(p).name)]
image_paths=sorted([os.path.join(FOLDER,x) for x in os.listdir(FOLDER) if Path(x).suffix.lower() in {'.png','.jpg','.jpeg','.webp'} and not x.startswith('_')],key=natural_key)
VALID_IMAGES=[]
for p in image_paths:
    try:
        with Image.open(p) as im: im.verify()
        with Image.open(p) as im: VALID_IMAGES.append({'path':p,'name':Path(p).name,'size':im.size})
    except Exception as e: print('⚠️ skipped',Path(p).name,e)
if not VALID_IMAGES: raise RuntimeError('No valid images in AIgenerated/.')
print('Images available:',len(VALID_IMAGES))
for i,x in enumerate(VALID_IMAGES,1): print(f'{i:02d}. {x["name"]} {x["size"][0]}x{x["size"][1]}')

Images available: 4
01. Cozy Afternoon on the Sofa.jpg 2172x1629
02. Cozy Floral Loungewear on the Sofa.jpg 1629x2172
03. Cozy Living Room Portrait with Purple Blooms.jpg 2172x1629
04. Playful Nighttime Window Portrait.jpg 1629x2172


In [ ]:

# **Cell 4**
USE_COUNT=min(len(VALID_IMAGES),len(LINES))
if len(VALID_IMAGES)<len(LINES):
    print(f'⚠️ {len(LINES)-len(VALID_IMAGES)} final lyric line(s) will be discarded because there are fewer images.')
    for z in LINES[len(VALID_IMAGES):]: print('  discarded:',z.get('line',z))
elif len(VALID_IMAGES)>len(LINES): print(f'ℹ️ {len(VALID_IMAGES)-len(LINES)} extra image(s) ignored.')
MAPPED=[]
for i in range(USE_COUNT):
    ln=LINES[i]; im=VALID_IMAGES[i]
    MAPPED.append({'slot':i+1,'line':ln,'image_path':im['path'],'image_name':im['name']})
print('Mapped image slots:',len(MAPPED))
for x in MAPPED: print(f'{x["slot"]:02d}. {x["image_name"]} → {x["line"].get("line","")}')

Mapped image slots: 4
01. Cozy Afternoon on the Sofa.jpg → Jaan 💖 ki qurbaani
02. Cozy Floral Loungewear on the Sofa.jpg → Le le dilbar 🤩 jaani
03. Cozy Living Room Portrait with Purple Blooms.jpg → Tabaahi pakki 👍 hai
04. Playful Nighttime Window Portrait.jpg → Aag 🔥 tu main 💧paani


In [ ]:

# **Cell 5**
MAP_START=float(LINES[0]['start'])
TIMELINE=[]
for x in MAPPED:
    ln=x['line']; s=float(ln['start'])-MAP_START; e=float(ln['end'])-MAP_START
    TIMELINE.append({**x,'start':max(0,s),'end':max(s,e),'duration':max(0,e-s)})
print('IMAGE TIMELINE')
for x in TIMELINE: print(f'{x["slot"]:02d}. {x["start"]:.3f} → {x["end"]:.3f} ({x["duration"]:.3f}s) | {x["image_name"]}')

IMAGE TIMELINE
01. 0.000 → 2.360 (2.360s) | Cozy Afternoon on the Sofa.jpg
02. 2.960 → 4.420 (1.460s) | Cozy Floral Loungewear on the Sofa.jpg
03. 4.940 → 6.680 (1.740s) | Cozy Living Room Portrait with Purple Blooms.jpg
04. 7.420 → 8.560 (1.140s) | Playful Nighttime Window Portrait.jpg


In [ ]:

# **Cell 6**
RNG=random.SystemRandom()
ANIMATIONS=['subtle_zoom_in','subtle_zoom_out','pan_left','pan_right','pan_up','pan_down','micro_drift']
for x in TIMELINE: x['animation']=RNG.choice(ANIMATIONS)
print('Animation plan ready; animation choice is per image, not per frame.')
for x in TIMELINE: print(x['image_name'],x['animation'])

Animation plan ready; animation choice is per image, not per frame.
Cozy Afternoon on the Sofa.jpg micro_drift
Cozy Floral Loungewear on the Sofa.jpg pan_down
Cozy Living Room Portrait with Purple Blooms.jpg pan_down
Playful Nighttime Window Portrait.jpg pan_up


In [ ]:

# **Cell 7**
OUTPUT_W,OUTPUT_H=1080,1920
FPS=30
print(f'Output: {OUTPUT_W}x{OUTPUT_H} @ {FPS}fps')
print('Policy: crop to 9:16; never non-uniformly stretch images.')

Output: 1080x1920 @ 30fps
Policy: crop to 9:16; never non-uniformly stretch images.


In [ ]:

# **Cell 8**
LYRIC_MODE='LINE'
print('Lyric mode:',LYRIC_MODE)

Lyric mode: LINE


In [ ]:

# **Cell 9**
PLAN_PATH=os.path.join(WORK_ROOT,'stage2_render_plan.json')
plan={'version':2,'audio':AUDIO_PATH,'song_map':SONG_MAP_PATH,'width':OUTPUT_W,'height':OUTPUT_H,'fps':FPS,'lyric_mode':LYRIC_MODE,'timeline':TIMELINE}
with open(PLAN_PATH,'w',encoding='utf-8') as f: json.dump(plan,f,ensure_ascii=False,indent=2)
print('✅ Render plan saved:',PLAN_PATH)

✅ Render plan saved: /content/drive/MyDrive/AIgenerated/_stage2_work/stage2_render_plan.json


In [ ]:

# **Cell 10**
print('STAGE 2 PART 1 CHECK'); print('====================')
print('Lyrics lines:',len(LINES)); print('Images:',len(VALID_IMAGES)); print('Mapped:',len(TIMELINE)); print('Audio:',f'{AUDIO_DURATION:.3f}s'); print('Mode:',LYRIC_MODE)
print('✅ Inputs ready.')

STAGE 2 PART 1 CHECK
Lyrics lines: 4
Images: 4
Mapped: 4
Audio: 9.060s
Mode: WORD
✅ Inputs ready.


In [ ]:

# **Cell 10B**
# ============================================================
# 🔄 RESUME MODE — RUN ONLY AS RESUMING
# ============================================================
# Use ONLY when:
# • Stage 1 is already complete.
# • trimmed_audio.m4a and song_map.json already exist in AIgenerated/.
# • You are starting a fresh runtime after disconnect/restart.
# • You want to resume Stage 2 without rerunning Cells 1–10.
#
# Do NOT use this for a brand-new Stage 2 project.

from google.colab import drive
import os, re, json, math, random, subprocess, glob
from pathlib import Path
from PIL import Image

drive.mount('/content/drive', force_remount=False)

FOLDER='/content/drive/MyDrive/AIgenerated'
AUDIO_PATH=os.path.join(FOLDER,'trimmed_audio.m4a')
SONG_MAP_PATH=os.path.join(FOLDER,'song_map.json')
ASSETS_ROOT=os.path.join(FOLDER,'assets')
FONTS_ROOT=os.path.join(ASSETS_ROOT,'fonts')
EMOJI_ROOT=os.path.join(ASSETS_ROOT,'emoji')
OUTPUT_ROOT=os.path.join(FOLDER,'stage2_outputs')
WORK_ROOT=os.path.join(FOLDER,'_stage2_work')

os.makedirs(OUTPUT_ROOT,exist_ok=True)
os.makedirs(WORK_ROOT,exist_ok=True)

if not os.path.isfile(AUDIO_PATH):
    raise FileNotFoundError(f'❌ Missing Stage 1 audio: {AUDIO_PATH}')
if not os.path.isfile(SONG_MAP_PATH):
    raise FileNotFoundError(f'❌ Missing Stage 1 song map: {SONG_MAP_PATH}')

with open(SONG_MAP_PATH,'r',encoding='utf-8') as f:
    SONG_MAP=json.load(f)

LINES=SONG_MAP.get('lines',[])
if not LINES:
    raise RuntimeError('❌ song_map.json contains no lyric lines.')

probe=subprocess.run(
    ['ffprobe','-v','error','-show_entries','format=duration',
     '-of','default=noprint_wrappers=1:nokey=1',AUDIO_PATH],
    capture_output=True,text=True,check=True
)
AUDIO_DURATION=float(probe.stdout.strip())

# Same image ordering used by normal Stage 2 Cell 3.
def natural_key(p):
    return [int(x) if x.isdigit() else x.lower()
            for x in re.split(r'(\d+)',Path(p).name)]

image_paths=sorted(
    [os.path.join(FOLDER,x) for x in os.listdir(FOLDER)
     if Path(x).suffix.lower() in {'.png','.jpg','.jpeg','.webp'}
     and not x.startswith('_')],
    key=natural_key
)

VALID_IMAGES=[]
for p in image_paths:
    try:
        with Image.open(p) as im:
            im.verify()
        with Image.open(p) as im:
            VALID_IMAGES.append({
                'path':p,
                'name':Path(p).name,
                'size':im.size
            })
    except Exception as e:
        print('⚠️ skipped',Path(p).name,e)

if not VALID_IMAGES:
    raise RuntimeError('❌ No valid images in AIgenerated/.')

USE_COUNT=min(len(VALID_IMAGES),len(LINES))

if len(VALID_IMAGES)<len(LINES):
    print(f'⚠️ {len(LINES)-len(VALID_IMAGES)} final lyric line(s) will be discarded because there are fewer images.')
elif len(VALID_IMAGES)>len(LINES):
    print(f'ℹ️ {len(VALID_IMAGES)-len(LINES)} extra image(s) ignored.')

MAPPED=[]
for i in range(USE_COUNT):
    ln=LINES[i]
    im=VALID_IMAGES[i]
    MAPPED.append({
        'slot':i+1,
        'line':ln,
        'image_path':im['path'],
        'image_name':im['name']
    })

MAP_START=float(LINES[0]['start'])
TIMELINE=[]
for x in MAPPED:
    ln=x['line']
    s=float(ln['start'])-MAP_START
    e=float(ln['end'])-MAP_START
    TIMELINE.append({
        **x,
        'start':max(0,s),
        'end':max(s,e),
        'duration':max(0,e-s)
    })

# Recreate the same Stage 2 rendering settings.
RNG=random.SystemRandom()
ANIMATIONS=[
    'subtle_zoom_in','subtle_zoom_out',
    'pan_left','pan_right','pan_up','pan_down','micro_drift'
]
for x in TIMELINE:
    x['animation']=RNG.choice(ANIMATIONS)

OUTPUT_W,OUTPUT_H=1080,1920
FPS=30
LYRIC_MODE='WORD'

print()
print('🔄 STAGE 2 RESUME MODE')
print('======================')
print('Lines:',len(LINES))
print('Images:',len(VALID_IMAGES))
print('Mapped:',len(TIMELINE))
print(f'Audio duration: {AUDIO_DURATION:.3f}s')
print('Lyric mode:',LYRIC_MODE)
print('✅ Stage 2 variables restored. Continue with Cell 11.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

🔄 STAGE 2 RESUME MODE
Lines: 4
Images: 4
Mapped: 4
Audio duration: 11.980s
Lyric mode: WORD
✅ Stage 2 variables restored. Continue with Cell 11.


In [ ]:

# **Cell 11**
BASE_VIDEO=os.path.join(OUTPUT_ROOT,'stage2_base.mp4')
if os.path.isfile(BASE_VIDEO):
    print('♻️ Existing base video found; reusing:',BASE_VIDEO)
else:
    def prep(im,t,dur,anim):
        im=im.convert('RGB'); sw,sh=im.size; tr=OUTPUT_W/OUTPUT_H; sr=sw/sh
        if sr>tr:
            cw=int(sh*tr); l=(sw-cw)//2; im=im.crop((l,0,l+cw,sh))
        else:
            ch=int(sw/tr); top=(sh-ch)//2; im=im.crop((0,top,sw,top+ch))
        im=im.resize((OUTPUT_W,OUTPUT_H),Image.Resampling.LANCZOS)
        p=max(0,min(1,t/max(dur,.001))); scale=1
        if anim=='subtle_zoom_in': scale=1+.035*p
        elif anim=='subtle_zoom_out': scale=1.035-.035*p
        elif anim in ('pan_left','pan_right','pan_up','pan_down','micro_drift'): scale=1.025
        if scale!=1:
            nw,nh=int(OUTPUT_W*scale),int(OUTPUT_H*scale); im=im.resize((nw,nh),Image.Resampling.LANCZOS)
            if anim=='pan_left': x=int((nw-OUTPUT_W)*p)
            elif anim=='pan_right': x=int((nw-OUTPUT_W)*(1-p))
            else: x=(nw-OUTPUT_W)//2
            if anim=='pan_up': y=int((nh-OUTPUT_H)*p)
            elif anim=='pan_down': y=int((nh-OUTPUT_H)*(1-p))
            else: y=(nh-OUTPUT_H)//2
            im=im.crop((x,y,x+OUTPUT_W,y+OUTPUT_H))
        return im
    cmd=['ffmpeg','-y','-f','rawvideo','-pix_fmt','rgb24','-s',f'{OUTPUT_W}x{OUTPUT_H}','-r',str(FPS),'-i','pipe:0','-an','-c:v','libx264','-preset','veryfast','-crf','20','-pix_fmt','yuv420p',BASE_VIDEO]
    p=subprocess.Popen(cmd,stdin=subprocess.PIPE,stdout=subprocess.DEVNULL,stderr=subprocess.PIPE)
    try:
        for x in TIMELINE:
            with Image.open(x['image_path']) as src: src=src.copy()
            n=max(1,round(x['duration']*FPS))
            for i in range(n): p.stdin.write(prep(src,i/max(1,n-1),x['duration'],x['animation']).tobytes())
        p.stdin.close(); err=p.stderr.read().decode(errors='ignore'); rc=p.wait()
        if rc: raise RuntimeError(err[-3000:])
    finally:
        try:p.stdin.close()
        except:pass
    print('✅ Base video saved:',BASE_VIDEO)
    print('No per-frame files created.')

✅ Base video saved: /content/drive/MyDrive/AIgenerated/stage2_outputs/stage2_base.mp4
No per-frame files created.


In [ ]:

# **Cell 11B**
# ⚡ RESUME MODE — RUN ONLY AS RESUMING
# Use ONLY when stage2_outputs/stage2_base.mp4 already exists.
# This cell is FULLY STANDALONE after Drive mount.
# It rebuilds every RAM variable required by Cell 13 and does NOT render the base video.

from google.colab import drive
import os,re,json,math,random,subprocess,glob
from pathlib import Path
from PIL import Image

drive.mount('/content/drive',force_remount=False)
FOLDER='/content/drive/MyDrive/AIgenerated'
AUDIO_PATH=os.path.join(FOLDER,'trimmed_audio.m4a')
SONG_MAP_PATH=os.path.join(FOLDER,'song_map.json')
ASSETS_ROOT=os.path.join(FOLDER,'assets')
FONTS_ROOT=os.path.join(ASSETS_ROOT,'fonts')
EMOJI_ROOT=os.path.join(ASSETS_ROOT,'emoji')
OUTPUT_ROOT=os.path.join(FOLDER,'stage2_outputs')
WORK_ROOT=os.path.join(FOLDER,'_stage2_work')
BASE_VIDEO=os.path.join(OUTPUT_ROOT,'stage2_base.mp4')
os.makedirs(OUTPUT_ROOT,exist_ok=True); os.makedirs(WORK_ROOT,exist_ok=True)

if not os.path.isfile(AUDIO_PATH): raise FileNotFoundError(f'❌ Missing Stage 1 audio: {AUDIO_PATH}')
if not os.path.isfile(SONG_MAP_PATH): raise FileNotFoundError(f'❌ Missing Stage 1 song map: {SONG_MAP_PATH}')
if not os.path.isfile(BASE_VIDEO): raise FileNotFoundError('❌ stage2_base.mp4 not found. Use Cell 10B → Cell 11 → Cell 12 instead.')

with open(SONG_MAP_PATH,'r',encoding='utf-8') as f: SONG_MAP=json.load(f)
LINES=SONG_MAP.get('lines',[])
if not LINES: raise RuntimeError('❌ song_map.json contains no lyric lines.')
probe=subprocess.run(['ffprobe','-v','error','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',AUDIO_PATH],capture_output=True,text=True,check=True)
AUDIO_DURATION=float(probe.stdout.strip())

def natural_key(p): return [int(x) if x.isdigit() else x.lower() for x in re.split(r'(\d+)',Path(p).name)]
image_paths=sorted([os.path.join(FOLDER,x) for x in os.listdir(FOLDER) if Path(x).suffix.lower() in {'.png','.jpg','.jpeg','.webp'} and not x.startswith('_')],key=natural_key)
VALID_IMAGES=[]
for p in image_paths:
    try:
        with Image.open(p) as im: im.verify()
        with Image.open(p) as im: VALID_IMAGES.append({'path':p,'name':Path(p).name,'size':im.size})
    except Exception as e: print('⚠️ skipped',Path(p).name,e)
if not VALID_IMAGES: raise RuntimeError('❌ No valid images in AIgenerated/.')

USE_COUNT=min(len(VALID_IMAGES),len(LINES))
if len(VALID_IMAGES)<len(LINES): print(f'⚠️ {len(LINES)-len(VALID_IMAGES)} final lyric line(s) will be discarded because there are fewer images.')
elif len(VALID_IMAGES)>len(LINES): print(f'ℹ️ {len(VALID_IMAGES)-len(LINES)} extra image(s) ignored.')
MAPPED=[]
for i in range(USE_COUNT): MAPPED.append({'slot':i+1,'line':LINES[i],'image_path':VALID_IMAGES[i]['path'],'image_name':VALID_IMAGES[i]['name']})
MAP_START=float(LINES[0]['start'])
TIMELINE=[]
for x in MAPPED:
    ln=x['line']; s=float(ln['start'])-MAP_START; e=float(ln['end'])-MAP_START
    TIMELINE.append({**x,'start':max(0,s),'end':max(s,e),'duration':max(0,e-s)})

OUTPUT_W,OUTPUT_H=1080,1920
FPS=30
LYRIC_MODE = globals().get('LYRIC_MODE', 'WORD')

base_probe=subprocess.run(['ffprobe','-v','error','-show_entries','format=duration:stream=width,height','-of','json',BASE_VIDEO],capture_output=True,text=True,check=True)
BASE_META=json.loads(base_probe.stdout)

print()
print('⚡ STAGE 2 RESUME — EXISTING BASE')
print('=================================')
print('Lines:',len(LINES))
print('Images:',len(VALID_IMAGES))
print('Mapped:',len(TIMELINE))
print(f'Audio duration: {AUDIO_DURATION:.3f}s')
print('Base video:',BASE_VIDEO)
print('Base metadata:')
print(json.dumps(BASE_META,indent=2))
print('Lyric mode:',LYRIC_MODE)
print('♻️ Existing base video will be reused.')
print('✅ Cell 11B complete. Continue directly to Cell 13.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

⚡ STAGE 2 RESUME — EXISTING BASE
Lines: 4
Images: 4
Mapped: 4
Audio duration: 9.060s
Base video: /content/drive/MyDrive/AIgenerated/stage2_outputs/stage2_base.mp4
Base metadata:
{
  "programs": [],
  "streams": [
    {
      "width": 1080,
      "height": 1920
    }
  ],
  "format": {
    "duration": "6.700000"
  }
}
Lyric mode: LINE
♻️ Existing base video will be reused.
✅ Cell 11B complete. Continue directly to Cell 13.


In [ ]:

# **Cell 12**
probe=subprocess.run(['ffprobe','-v','error','-show_entries','format=duration:stream=width,height','-of','json',BASE_VIDEO],capture_output=True,text=True,check=True)
print('BASE VIDEO CHECK'); print(json.dumps(json.loads(probe.stdout),indent=2)); print('✅ Base checkpoint valid.')

BASE VIDEO CHECK
{
  "programs": [],
  "streams": [
    {
      "width": 1080,
      "height": 1920
    }
  ],
  "format": {
    "duration": "6.700000"
  }
}
✅ Base checkpoint valid.


In [ ]:

# **Cell 13**
# ============================================================
# 🎵 LYRIC RENDERER — MINIMAL LINE + WORD SUPPORT
# ============================================================
# Original working WORD renderer. Only event-building supports LINE.
# Everything else is unchanged except the final pipe receives RGB,
# because FFmpeg rawvideo is configured as rgb24.

from PIL import Image, ImageDraw, ImageFont
import os, re, glob, json, math, random, subprocess
from pathlib import Path

LYRIC_VIDEO=os.path.join(OUTPUT_ROOT,'stage2_lyrics.mp4')
os.makedirs(WORK_ROOT,exist_ok=True)

FONT_FILES=[]
for root,ds,fs in os.walk(FONTS_ROOT):
    FONT_FILES += [os.path.join(root,f) for f in fs if f.lower().endswith(('.ttf','.otf'))]
if not FONT_FILES:
    FONT_FILES=glob.glob('/usr/share/fonts/**/*.ttf',recursive=True)
if not FONT_FILES:
    raise RuntimeError('No font files found.')

EMOJI_FILES=[]
for root,ds,fs in os.walk(EMOJI_ROOT):
    EMOJI_FILES += [os.path.join(root,f) for f in fs if f.lower().endswith(('.png','.webp'))]

print(f'Fonts available: {len(FONT_FILES)}')
print(f'Emoji assets available: {len(EMOJI_FILES)}')
print('Lyric mode:',LYRIC_MODE)

def emoji_path(ch):
    clean=[c for c in ch if ord(c)!=0xfe0f and not 0x1f3fb<=ord(c)<=0x1f3ff]
    cp='-'.join(format(ord(c),'x') for c in clean)
    for p in EMOJI_FILES:
        stem=Path(p).stem.lower()
        if stem==cp or stem.startswith(cp+'-'):
            return p
    return None

def split_emoji(text):
    chars=list(text); found=[]
    while chars and (ord(chars[-1])>0x1f000 or chars[-1] in '❤❌⭐✨☁🌺🌻🌹🌷💞💫'):
        found.insert(0,chars.pop())
        if chars and chars[-1]=='️':
            found.insert(0,chars.pop())
    return ''.join(chars).rstrip(),''.join(found)

def choose_font(size):
    random.shuffle(FONT_FILES)
    for f in FONT_FILES[:20]:
        try:return ImageFont.truetype(f,size)
        except:pass
    return ImageFont.truetype(FONT_FILES[0],size)

# ------------------------------------------------------------
# BUILD EVENTS — ONLY LINE/WORD DIFFERENCE
# ------------------------------------------------------------
MAP_START=float(LINES[0]['start'])
word_events=[]

if str(LYRIC_MODE).upper()=='LINE':
    for li in LINES[:len(TIMELINE)]:
        plain,emoji=split_emoji(str(li.get('line','')))
        word_events.append({
            'start':max(0.0,float(li['start'])-MAP_START),
            'end':max(0.0,float(li['end'])-MAP_START),
            'text':plain,
            'emoji':emoji
        })
else:
    for li in LINES[:len(TIMELINE)]:
        for w in li.get('words',[]):
            raw=str(w.get('word','')).strip()
            if not raw:
                continue
            ws=float(w['start'])-MAP_START
            we=float(w['end'])-MAP_START
            if we>0:
                word_events.append({'start':max(0.0,ws),'end':max(ws,we),'text':raw})

    for li in LINES[:len(TIMELINE)]:
        words=[w for w in li.get('words',[]) if str(w.get('word','')).strip()]
        plain,emoji=split_emoji(str(li.get('line','')))
        if emoji and words:
            target=words[-1]
            target_text=str(target.get('word','')).strip()
            target_start=float(target['start'])-MAP_START
            for ev in word_events:
                if ev['text']==target_text and abs(ev['start']-target_start)<.02:
                    ev['emoji']=emoji
                    break

styles={}
PALETTE=[(255,255,255),(255,80,180),(0,235,255),(255,225,70),(170,100,255),(90,255,130),(255,145,60)]
for ev in word_events:
    styles[(round(ev['start'],3),ev['text'])]={
        'size':random.choice([76,86,96,108,120]),
        'fill':random.choice(PALETTE),
        'stroke':random.choice([(0,0,0),(255,255,255),(40,20,60)]),
        'sw':random.choice([2,3,4]),
        'x':random.randint(250,830),
        'y':random.choice([330,620,900,1180,1450]),
        'font':choose_font(random.choice([76,86,96,108,120]))
    }

def draw_event(frame,ev,progress):
    st=styles[(round(ev['start'],3),ev['text'])]
    text=ev['text']; emoji=ev.get('emoji','')
    reveal=max(1,min(len(text),int(math.ceil(len(text)*min(1,progress/.12))))) if text else 0
    shown=text[:reveal]
    layer=Image.new('RGBA',frame.size,(0,0,0,0)); d=ImageDraw.Draw(layer); font=st['font']
    bbox=d.textbbox((0,0),shown,font=font,stroke_width=st['sw']); tw=bbox[2]-bbox[0]
    x=max(20,min(OUTPUT_W-tw-20,st['x'])); y=st['y']
    d.text((x+3,y+3),shown,font=font,fill=(0,0,0,130),stroke_width=st['sw']+1,stroke_fill=(0,0,0,100))
    d.text((x,y),shown,font=font,fill=st['fill'],stroke_width=st['sw'],stroke_fill=st['stroke'])
    if emoji:
        ep=emoji_path(emoji)
        if ep:
            try:
                with Image.open(ep) as ei:
                    ei=ei.convert('RGBA'); target=max(42,int(font.size*.78)); ei.thumbnail((target,target),Image.Resampling.LANCZOS)
                    ex=min(OUTPUT_W-ei.width-15,x+tw+10); ey=y+max(0,(font.size-ei.height)//2); layer.alpha_composite(ei,(ex,ey))
            except: pass
    return Image.alpha_composite(frame.convert('RGBA'),layer)

# ------------------------------------------------------------
# ORIGINAL FFMPEG PIPELINE — UNCHANGED
# ------------------------------------------------------------
reader=subprocess.Popen(['ffmpeg','-v','error','-i',BASE_VIDEO,'-f','rawvideo','-pix_fmt','rgb24','-r',str(FPS),'pipe:1'],stdout=subprocess.PIPE,stderr=subprocess.PIPE)
writer=subprocess.Popen(['ffmpeg','-y','-f','rawvideo','-pix_fmt','rgb24','-s',f'{OUTPUT_W}x{OUTPUT_H}','-r',str(FPS),'-i','pipe:0','-i',AUDIO_PATH,'-map','0:v:0','-map','1:a:0','-c:v','libx264','-preset','veryfast','-crf','20','-pix_fmt','yuv420p','-c:a','aac','-b:a','192k','-t',str(AUDIO_DURATION),LYRIC_VIDEO],stdin=subprocess.PIPE,stdout=subprocess.DEVNULL,stderr=subprocess.PIPE)
fb=OUTPUT_W*OUTPUT_H*3; idx=0
try:
    while True:
        raw=reader.stdout.read(fb)
        if len(raw)<fb: break
        t=idx/FPS; frame=Image.frombytes('RGB',(OUTPUT_W,OUTPUT_H),raw)
        active=[e for e in word_events if e['start']<=t<e['end']]
        for ev in active:
            frame=draw_event(frame,ev,(t-ev['start'])/max(.001,ev['end']-ev['start']))
        # IMPORTANT: draw_event returns RGBA; FFmpeg pipe expects rgb24.
        writer.stdin.write(frame.convert('RGB').tobytes()); idx+=1
    reader.stdout.close(); reader.wait(); writer.stdin.close(); err=writer.stderr.read().decode(errors='ignore'); rc=writer.wait()
    if rc: raise RuntimeError('Lyric render failed:\n'+err[-3000:])
finally:
    try:reader.kill()
    except:pass
    try:writer.stdin.close()
    except:pass
print('✅ Lyric video saved:',LYRIC_VIDEO)
print('Mode:',LYRIC_MODE)
print('Events rendered:',len(word_events))
print('No per-frame/per-word PNG files created.')

Fonts available: 89
Emoji assets available: 3689
Lyric mode: LINE
✅ Lyric video saved: /content/drive/MyDrive/AIgenerated/stage2_outputs/stage2_lyrics.mp4
Mode: LINE
Events rendered: 4
No per-frame/per-word PNG files created.


In [ ]:
# FINAL VIDEO CHECK
# ============================================================
probe=subprocess.run(
    ['ffprobe','-v','error','-show_entries','format=duration:stream=width,height','-of','json',LYRIC_VIDEO],
    capture_output=True,text=True,check=True
)
print('FINAL VIDEO CHECK')
print('=================')
print(json.dumps(json.loads(probe.stdout),indent=2))
print('Mode:',LYRIC_MODE)
print('Final video:',LYRIC_VIDEO)
print('Audio source:',AUDIO_PATH)
print('Song map:',SONG_MAP_PATH)
print('✅ Final render exists.')

FINAL VIDEO CHECK
{
  "programs": [],
  "streams": [
    {
      "width": 1080,
      "height": 1920
    },
    {}
  ],
  "format": {
    "duration": "9.060000"
  }
}
Mode: LINE
Final video: /content/drive/MyDrive/AIgenerated/stage2_outputs/stage2_lyrics.mp4
Audio source: /content/drive/MyDrive/AIgenerated/trimmed_audio.m4a
Song map: /content/drive/MyDrive/AIgenerated/song_map.json
✅ Final render exists.


In [ ]:

# **Cell 15**

import os
import shutil
from pathlib import Path

# ============================================================
# USER SETTINGS — change only these if needed
# ============================================================
PROJECT_NAME = 'Chal Wahn jate hain 4lines'
KEEP_SONG_MP3 = False          # DEFAULT: NO
KEEP_BASE_VIDEO = False        # DEFAULT: NO

# ============================================================
# PATHS
# ============================================================
AI_ROOT = '/content/drive/MyDrive/AIgenerated'
ARCHIVE_ROOT = '/content/drive/MyDrive/Completed video sync'

os.makedirs(ARCHIVE_ROOT, exist_ok=True)

# ============================================================
# IMPORTANT PROJECT FILES
# ============================================================
source_files = [
    ('lyrics.txt', os.path.join(AI_ROOT, 'lyrics.txt'), True),
    ('trimmed_audio.m4a', os.path.join(AI_ROOT, 'trimmed_audio.m4a'), True),
    ('song_map.json', os.path.join(AI_ROOT, 'song_map.json'), True),
    ('stage2_lyrics.mp4', os.path.join(AI_ROOT, 'stage2_outputs', 'stage2_lyrics.mp4'), True),
    ('song.mp3', os.path.join(AI_ROOT, 'song.mp3'), KEEP_SONG_MP3),
    ('stage2_base.mp4', os.path.join(AI_ROOT, 'stage2_outputs', 'stage2_base.mp4'), KEEP_BASE_VIDEO),
]

# ============================================================
# FIND A FREE ARCHIVE FOLDER — NEVER OVERWRITE
# ============================================================
base_name = str(PROJECT_NAME).strip() or 'Project'
archive_dir = os.path.join(ARCHIVE_ROOT, base_name)

counter = 2
while os.path.exists(archive_dir):
    archive_dir = os.path.join(ARCHIVE_ROOT, f'{base_name}{counter}')
    counter += 1

os.makedirs(archive_dir)

# ============================================================
# COPY + VERIFY
# ============================================================
archived = []
missing = []

for filename, src, enabled in source_files:
    if not enabled:
        continue

    if not os.path.isfile(src):
        missing.append(filename)
        continue

    dst = os.path.join(archive_dir, filename)
    shutil.copy2(src, dst)

    # Basic size verification before source cleanup.
    if os.path.getsize(src) != os.path.getsize(dst):
        raise RuntimeError(f'❌ Verification failed for {filename}. Source was NOT deleted.')

    archived.append(filename)

# Required files must exist before we touch workspace files.
required_names = ['lyrics.txt', 'trimmed_audio.m4a', 'song_map.json', 'stage2_lyrics.mp4']
missing_required = [x for x in required_names if x not in archived]

if missing_required:
    # Remove incomplete archive rather than leaving a misleading project folder.
    shutil.rmtree(archive_dir, ignore_errors=True)
    raise RuntimeError(
        '❌ Archive aborted. Required file(s) missing: '
        + ', '.join(missing_required)
    )

# ============================================================
# REMOVE COPIED PROJECT FILES FROM ACTIVE WORKSPACE
# Keep assets/ untouched.
# ============================================================
for filename in archived:
    for _, src, enabled in source_files:
        if enabled and os.path.basename(src) == filename and os.path.isfile(src):
            os.remove(src)
            break

print('================================')
print('PROJECT ARCHIVED')
print('================================')
print('Archive folder:', archive_dir)
print('Saved files:')
for x in archived:
    print('  ✅', x)

print()
print('Not archived by default:')
if not KEEP_SONG_MP3:
    print('  ❌ song.mp3')
if not KEEP_BASE_VIDEO:
    print('  ❌ stage2_base.mp4')

print()
print('⚠️ Note: this cell archives only the important project files.')
print('Run Cell 16 separately if you want a complete AIgenerated workspace reset.')

In [ ]:

# **Cell 16**
# TOTAL CLEANUP
# Deletes current AIgenerated workspace files/folders,
# but preserves permanent assets/ and stage2_outputs/ folders.

from google.colab import drive
import os
import shutil

drive.mount('/content/drive', force_remount=False)

FOLDER = '/content/drive/MyDrive/AIgenerated'
ASSETS_ROOT = os.path.join(FOLDER, 'assets')
OUTPUT_ROOT = os.path.join(FOLDER, 'stage2_outputs')

if not os.path.isdir(ASSETS_ROOT):
    raise RuntimeError('❌ assets/ folder not found. Cleanup cancelled for safety.')

os.makedirs(OUTPUT_ROOT, exist_ok=True)

KEEP = {
    os.path.abspath(ASSETS_ROOT),
    os.path.abspath(OUTPUT_ROOT),
}

deleted = []

for name in os.listdir(FOLDER):

    path = os.path.abspath(os.path.join(FOLDER, name))

    # Keep permanent folders
    if path in KEEP:
        continue

    try:
        if os.path.isdir(path):
            shutil.rmtree(path)
        else:
            os.remove(path)

        deleted.append(name)

    except Exception as e:
        print(f'⚠️ Could not delete {name}: {e}')

print()
print('================================')
print('TOTAL CLEANUP COMPLETE')
print('================================')
print('Deleted:', len(deleted), 'items')
print('✅ Kept: assets/')
print('✅ Kept: stage2_outputs/ (folder only)')
print('🧹 AIgenerated workspace is clean.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

TOTAL CLEANUP COMPLETE
Deleted: 1 items
✅ Kept: assets/
✅ Kept: stage2_outputs/ (folder only)
🧹 AIgenerated workspace is clean.
